In [2]:
import pandas as pd
df =pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1',)
print(df.head())
print(df['sentiment'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/content/IMDB Dataset.csv'

In [ ]:
df

In [ ]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [ ]:
#preprocessing
import re
negation_words=[
    "not good","not bad","not great","don't like","didin't like","never liked", "wasn't good","isn't good","no good"

]
def clean_text(text):
  text=re.sub(r"[^a-zA-Z\s']","",text)
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [ ]:
#splitting
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)



In [ ]:
#model training
from tensorflow.keras.preprocessing.text import Tokenizer
from  tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size=20000
max_len=250

tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV>")#out of vocabulary
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test) # Corrected: use X_test here

X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

model=Sequential([
    Embedding(vocab_size,100),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3, return_sequences=True),
    LSTM(64,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.5),
    Dense(1,activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']

)
model.summary()


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history=model.fit(
    X_train_pad,y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
loss,acc=model.evaluate(X_test_pad,y_test)
print("Test Accuracy:",acc)

In [ ]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded_sequence=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview",review)
  print("Score:",prediction)

  if prediction>=0.5:
    print("Sentiment:Positive")
  else:
    print("Sentiment:Negative")

In [ ]:
predict_sentiment("Movie is really good and amazing")